In [3]:
import pandas as pd
from string import Template

In [4]:
### load data
baseline_df = pd.read_csv('Data/baseline_data.csv', parse_dates=['d.birth'])
diag_df = pd.read_csv('Data/diag_data.csv', parse_dates=['sample_date'])
snomed_df = pd.read_csv('Data/snomed_data.csv')

### resources templates
PATIENT_TEMPLATE = Template("""
{
  "resourceType": "Patient",
  "id": "$patient_id",
  "active": "$active",
  "gender": "$gender",
  "birthDate": "$birthdate"
}
""")

CONDITION_TEMPLATE = Template("""
{
  "resourceType": "Condition",
  "id": "$condition_id",
  "asserationDate": "$assertion_date",
  "subject": {
    "reference": "Patient/$patient_id"
  },
  "code": $codeable_concept,
  "clinicalStatus": $clinical_status_codeable
}
""")

CODEABLE_CONTENT_TEMPLATE = Template("""{ 
      "coding": [{
        "system": "$system",
        "code": "$code",
        "display": "$display"
      }],
    "text": "$text"
  }""")


def get_snomed_data(code_str):
    snomed_code_df = snomed_df.loc[snomed_df['label'] == code_str].iloc[0]
    code = snomed_code_df['code']
    display_text = snomed_code_df['text']
    return code, display_text


FileNotFoundError: [Errno 2] No such file or directory: 'Data/snomed_data.csv'

In [ ]:
### Patient
fhir_patient_resources = []
for index, row in baseline_df.iterrows():
    patient_data = {
        'patient_id': row['id'],
        'active': True,
        'gender': row['sex'],
        'birthdate': row['d.birth']
    }

    patient_resource = PATIENT_TEMPLATE.substitute(patient_data)
    fhir_patient_resources.append(patient_resource)
    break

for resource in fhir_patient_resources:
    print(resource)
    with open('patient.json', 'w', encoding='utf-8') as f:
        f.write(resource)

In [ ]:
### Condition
fhir_condition_resources = []
for index, row in diag_df.iterrows():
    snomed_data = get_snomed_data(row['code'])

    codeable_concept_data = {
        'system': 'http://snomed.info/sct',
        'code': snomed_data[0],
        'display': snomed_data[1],
        'text': snomed_data[1]
    }

    clinical_status_codeable_data = {
        'system': 'http://terminology.hl7.org/CodeSystem/clinical-status',
        'code': 'unknown',
        'display': 'Unknown',
        'text': 'Unknown'
    }

    codeable_concept_json = CODEABLE_CONTENT_TEMPLATE.substitute(codeable_concept_data)
    clinical_status_json = CODEABLE_CONTENT_TEMPLATE.substitute(clinical_status_codeable_data)

    condition_data = {
        'condition_id': index,
        'patient_id': row['id'],
        'assertion_date': row['sample_date'],
        'codeable_concept': codeable_concept_json,
        'clinical_status_codeable': clinical_status_json,
    }

    condition_resource = CONDITION_TEMPLATE.substitute(condition_data)
    fhir_condition_resources.append(condition_resource)
    break

for resource in fhir_condition_resources:
    print(resource)
    with open('condition.json', 'w', encoding='utf-8') as f:
        f.write(resource)
